In [1]:
import pandas as pd
from pathlib import Path

PROC_DIR = Path("..") / "data" / "processed"
df = pd.read_csv(PROC_DIR / "covid_clean.csv", parse_dates=["date"])

# Per-million metrics
df["cases_per_million"] = df["total_cases"] / df["population"] * 1_000_000
df["deaths_per_million"] = df["total_deaths"] / df["population"] * 1_000_000

# Case fatality rate
df["case_fatality_rate"] = df["total_deaths"] / df["total_cases"]
df["case_fatality_rate"] = df["case_fatality_rate"].replace([float("inf")], pd.NA)

# No vaccination data in this CSV
df["vaccination_rate"] = pd.NA

# Rolling 7-day metrics
df = df.sort_values(["location", "date"])
df["rolling_7d_cases"] = df.groupby("location")["new_cases"].transform(lambda x: x.rolling(7, min_periods=1).sum())
df["rolling_7d_deaths"] = df.groupby("location")["new_deaths"].transform(lambda x: x.rolling(7, min_periods=1).sum())

# Weekly aggregation
weekly = (
    df.set_index("date")
      .groupby("location")
      .resample("W")
      .agg({
          "iso_code": "first",
          "continent": "first",
          "population": "first",
          "new_cases": "sum",
          "new_deaths": "sum",
          "rolling_7d_cases": "mean",
          "rolling_7d_deaths": "mean",
          "cases_per_million": "max",
          "deaths_per_million": "max",
          "case_fatality_rate": "max"
      })
      .reset_index()
)

weekly_path = PROC_DIR / "covid_weekly_features.csv"
weekly.to_csv(weekly_path, index=False)
weekly_path

PosixPath('../data/processed/covid_weekly_features.csv')